# Model Error Analysis

Appendix notebook for reviewing LightGBM prediction errors and score behavior.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pred_path = ROOT / 'data' / 'processed' / 'day10_11_lightgbm_predictions.parquet'
predictions = pd.read_parquet(pred_path) if pred_path.exists() else pd.DataFrame()
predictions.shape

In [ ]:
if predictions.empty:
    display(pd.DataFrame())
else:
    by_model = predictions.groupby('model_name').agg(
        rows=('market_hash_name', 'size'),
        mean_score=('strong_buy_score', 'mean'),
        max_score=('strong_buy_score', 'max'),
    )
    display(by_model)

In [ ]:
if {'model_name', 'label_code', 'predicted_label_code'}.issubset(predictions.columns):
    correct = predictions['label_code'].eq(predictions['predicted_label_code'])
    errors = predictions.assign(correct=correct)
    display(errors.groupby('model_name')['correct'].mean().rename('accuracy_proxy'))
else:
    display(pd.Series(dtype=float, name='accuracy_proxy'))